# Data Lineage Tutorial

This notebook demonstrates how to query and visualize data lineage using the Marquez API.

## Contents
1. Connect to Marquez API
2. Query Namespaces and Jobs
3. Explore Datasets
4. Visualize Lineage Graphs
5. Track Column-Level Lineage
6. Perform Impact Analysis
7. Create Custom Lineage Events

In [ ]:
# Install required packages
!pip install requests pandas matplotlib networkx plotly openlineage-python

In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Marquez API configuration
MARQUEZ_URL = "http://marquez-api:5000/api/v1"

print(f"Marquez API URL: {MARQUEZ_URL}")

## 1. Connect to Marquez API

First, let's verify we can connect to the Marquez API and check its health.

In [ ]:
def check_marquez_health():
    """Check if Marquez API is accessible"""
    try:
        response = requests.get(f"{MARQUEZ_URL.replace('/api/v1', '')}/api/v1/health")
        if response.status_code == 200:
            print("✓ Marquez API is healthy")
            return True
        else:
            print(f"✗ Marquez API returned status {response.status_code}")
            return False
    except Exception as e:
        print(f"✗ Could not connect to Marquez: {str(e)}")
        return False

check_marquez_health()

## 2. Query Namespaces and Jobs

Let's explore what namespaces and jobs exist in Marquez.

In [ ]:
def get_namespaces():
    """Get all namespaces"""
    response = requests.get(f"{MARQUEZ_URL}/namespaces")
    if response.status_code == 200:
        data = response.json()
        namespaces = [ns['name'] for ns in data.get('namespaces', [])]
        return namespaces
    return []

def get_jobs(namespace):
    """Get all jobs in a namespace"""
    response = requests.get(f"{MARQUEZ_URL}/namespaces/{namespace}/jobs")
    if response.status_code == 200:
        return response.json().get('jobs', [])
    return []

# Get and display namespaces
namespaces = get_namespaces()
print(f"Found {len(namespaces)} namespace(s):")
for ns in namespaces:
    print(f"  - {ns}")
    jobs = get_jobs(ns)
    print(f"    Jobs: {len(jobs)}")
    for job in jobs[:5]:  # Show first 5 jobs
        print(f"      • {job.get('name', 'Unknown')}")
    if len(jobs) > 5:
        print(f"      ... and {len(jobs) - 5} more")

## 3. Explore Datasets

Let's look at the datasets being tracked.

In [ ]:
def get_datasets(namespace):
    """Get all datasets in a namespace"""
    response = requests.get(f"{MARQUEZ_URL}/namespaces/{namespace}/datasets")
    if response.status_code == 200:
        return response.json().get('datasets', [])
    return []

def get_dataset_details(namespace, dataset_name):
    """Get detailed information about a dataset"""
    from urllib.parse import quote
    response = requests.get(f"{MARQUEZ_URL}/namespaces/{namespace}/datasets/{quote(dataset_name, safe='')}")
    if response.status_code == 200:
        return response.json()
    return None

# Explore datasets
if namespaces:
    namespace = namespaces[0]  # Use first namespace
    datasets = get_datasets(namespace)
    print(f"\nDatasets in '{namespace}':")
    print(f"Total: {len(datasets)}\n")
    
    # Display as DataFrame
    if datasets:
        df_datasets = pd.DataFrame([
            {
                'Name': ds.get('name', 'Unknown'),
                'Type': ds.get('type', 'Unknown'),
                'Source': ds.get('sourceName', 'Unknown'),
                'Updated': ds.get('updatedAt', 'Unknown')
            }
            for ds in datasets[:10]  # Show first 10
        ])
        display(df_datasets)
    else:
        print("No datasets found. Run some Spark jobs or Airflow DAGs to generate lineage!")

## 4. Visualize Lineage Graphs

Let's visualize the lineage graph for a specific job or dataset.

In [ ]:
def get_lineage(namespace, node_type, node_name, depth=10):
    """Get lineage graph for a node"""
    from urllib.parse import quote
    node_id = f"{node_type}:{namespace}:{node_name}"
    response = requests.get(
        f"{MARQUEZ_URL}/lineage",
        params={'nodeId': node_id, 'depth': depth}
    )
    if response.status_code == 200:
        return response.json()
    return None

def visualize_lineage(lineage_data):
    """Visualize lineage graph using networkx and matplotlib"""
    if not lineage_data or 'graph' not in lineage_data:
        print("No lineage data to visualize")
        return
    
    # Create directed graph
    G = nx.DiGraph()
    
    # Add nodes
    for node in lineage_data['graph']:
        node_type = node.get('type', 'UNKNOWN')
        node_id = node.get('id', '')
        node_data = node.get('data', {})
        
        G.add_node(
            node_id,
            type=node_type,
            label=node_data.get('name', node_id),
            data=node_data
        )
    
    # Add edges
    for node in lineage_data['graph']:
        node_id = node.get('id', '')
        for in_edge in node.get('inEdges', []):
            G.add_edge(in_edge.get('origin', ''), node_id)
        for out_edge in node.get('outEdges', []):
            G.add_edge(node_id, out_edge.get('destination', ''))
    
    # Create visualization
    plt.figure(figsize=(15, 10))
    pos = nx.spring_layout(G, k=2, iterations=50)
    
    # Color nodes by type
    color_map = {
        'DATASET': 'lightblue',
        'JOB': 'lightgreen',
        'UNKNOWN': 'lightgray'
    }
    
    node_colors = [color_map.get(G.nodes[node].get('type', 'UNKNOWN'), 'lightgray') for node in G.nodes()]
    node_labels = {node: G.nodes[node].get('label', node)[:30] for node in G.nodes()}
    
    nx.draw(G, pos, 
            node_color=node_colors,
            node_size=3000,
            with_labels=True,
            labels=node_labels,
            font_size=8,
            font_weight='bold',
            arrows=True,
            arrowsize=20,
            edge_color='gray',
            linewidths=2,
            arrowstyle='->')
    
    plt.title("Data Lineage Graph", fontsize=16, fontweight='bold')
    plt.axis('off')
    
    # Add legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightblue', markersize=15, label='Dataset'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightgreen', markersize=15, label='Job')
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nGraph Statistics:")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")
    print(f"  Datasets: {sum(1 for n in G.nodes() if G.nodes[n].get('type') == 'DATASET')}")
    print(f"  Jobs: {sum(1 for n in G.nodes() if G.nodes[n].get('type') == 'JOB')}")

# Example: Get lineage for a job
if namespaces and jobs:
    namespace = namespaces[0]
    jobs = get_jobs(namespace)
    if jobs:
        job_name = jobs[0].get('name', '')
        print(f"Getting lineage for job: {job_name}")
        lineage = get_lineage(namespace, 'job', job_name)
        visualize_lineage(lineage)
else:
    print("No jobs found to visualize. Run example Spark jobs or Airflow DAGs first!")

## 5. Track Column-Level Lineage

Explore how columns are derived from source to destination.

In [ ]:
def get_column_lineage(dataset_details):
    """Extract column lineage from dataset details"""
    if not dataset_details:
        return None
    
    fields = dataset_details.get('fields', [])
    column_lineage = []
    
    for field in fields:
        field_name = field.get('name', 'Unknown')
        field_type = field.get('type', 'Unknown')
        
        # Check for column lineage facet
        input_fields = field.get('inputFields', [])
        transformation = field.get('transformationType', 'DIRECT')
        
        column_lineage.append({
            'column': field_name,
            'type': field_type,
            'transformation': transformation,
            'source_columns': [f"{f.get('namespace', '')}:{f.get('name', '')}:{f.get('field', '')}" 
                              for f in input_fields]
        })
    
    return pd.DataFrame(column_lineage)

# Example: Get column lineage for a dataset
if namespaces:
    namespace = namespaces[0]
    datasets = get_datasets(namespace)
    if datasets:
        dataset_name = datasets[0].get('name', '')
        print(f"Column lineage for: {dataset_name}\n")
        dataset_details = get_dataset_details(namespace, dataset_name)
        if dataset_details:
            col_lineage = get_column_lineage(dataset_details)
            if col_lineage is not None and not col_lineage.empty:
                display(col_lineage)
            else:
                print("No column lineage information available for this dataset")
        else:
            print("Could not retrieve dataset details")
    else:
        print("No datasets found")
else:
    print("No namespaces found")

## 6. Perform Impact Analysis

Find all downstream datasets and jobs affected by a change.

In [ ]:
def impact_analysis(namespace, dataset_name, depth=20):
    """Analyze the impact of changing a dataset"""
    print(f"Impact Analysis for: {dataset_name}\n")
    print("="*80)
    
    # Get lineage
    lineage = get_lineage(namespace, 'dataset', dataset_name, depth)
    
    if not lineage or 'graph' not in lineage:
        print("No lineage data available")
        return
    
    affected_jobs = set()
    affected_datasets = set()
    
    for node in lineage['graph']:
        node_type = node.get('type', '')
        node_data = node.get('data', {})
        node_name = node_data.get('name', 'Unknown')
        
        if node_type == 'JOB' and node_name != dataset_name:
            affected_jobs.add(node_name)
        elif node_type == 'DATASET' and node_name != dataset_name:
            affected_datasets.add(node_name)
    
    print(f"\n📊 Impact Summary:")
    print(f"  • {len(affected_jobs)} jobs will be affected")
    print(f"  • {len(affected_datasets)} datasets will be impacted")
    
    if affected_jobs:
        print(f"\n🔧 Affected Jobs:")
        for i, job in enumerate(sorted(affected_jobs)[:10], 1):
            print(f"  {i}. {job}")
        if len(affected_jobs) > 10:
            print(f"  ... and {len(affected_jobs) - 10} more")
    
    if affected_datasets:
        print(f"\n📁 Affected Datasets:")
        for i, ds in enumerate(sorted(affected_datasets)[:10], 1):
            print(f"  {i}. {ds}")
        if len(affected_datasets) > 10:
            print(f"  ... and {len(affected_datasets) - 10} more")
    
    print("\n" + "="*80)
    print("\n⚠️  Recommendation:")
    print("  1. Review all affected jobs and datasets")
    print("  2. Test changes in a development environment")
    print("  3. Update downstream jobs if schema changes")
    print("  4. Communicate changes to data consumers")
    print("  5. Monitor data quality after changes")
    
    return {
        'affected_jobs': list(affected_jobs),
        'affected_datasets': list(affected_datasets)
    }

# Example impact analysis
if namespaces:
    namespace = namespaces[0]
    datasets = get_datasets(namespace)
    if datasets:
        dataset_name = datasets[0].get('name', '')
        impact = impact_analysis(namespace, dataset_name)
    else:
        print("No datasets found for impact analysis")
else:
    print("No namespaces found")

## 7. Create Custom Lineage Events

Learn how to programmatically create lineage events using the OpenLineage Python client.

In [ ]:
from openlineage.client import OpenLineageClient
from openlineage.client.run import RunEvent, RunState, Run, Job
from openlineage.client.facet import SqlJobFacet, SourceCodeLocationJobFacet
import uuid

# Initialize OpenLineage client
client = OpenLineageClient(url="http://marquez-api:5000")

def create_sample_lineage_event():
    """Create a sample lineage event"""
    
    # Define job
    job = Job(
        namespace="datalake",
        name="notebook_custom_job"
    )
    
    # Define run
    run = Run(runId=str(uuid.uuid4()))
    
    # Define inputs
    inputs = [
        {
            "namespace": "datalake",
            "name": "s3://data-lake/input/sample.csv",
            "facets": {
                "schema": {
                    "_producer": "notebook",
                    "_schemaURL": "https://openlineage.io/spec/facets/1-0-0/SchemaDatasetFacet.json",
                    "fields": [
                        {"name": "id", "type": "INTEGER"},
                        {"name": "name", "type": "STRING"},
                        {"name": "value", "type": "DOUBLE"}
                    ]
                }
            }
        }
    ]
    
    # Define outputs
    outputs = [
        {
            "namespace": "datalake",
            "name": "s3://data-lake/output/processed.parquet",
            "facets": {
                "schema": {
                    "_producer": "notebook",
                    "_schemaURL": "https://openlineage.io/spec/facets/1-0-0/SchemaDatasetFacet.json",
                    "fields": [
                        {"name": "id", "type": "INTEGER"},
                        {"name": "name", "type": "STRING"},
                        {"name": "processed_value", "type": "DOUBLE"},
                        {"name": "timestamp", "type": "TIMESTAMP"}
                    ]
                }
            }
        }
    ]
    
    # Send START event
    print("Sending START event...")
    start_event = RunEvent(
        eventType=RunState.START,
        eventTime=datetime.now().isoformat(),
        run=run,
        job=job,
        producer="jupyter-notebook",
        inputs=inputs,
        outputs=[]
    )
    client.emit(start_event)
    
    # Simulate some work
    print("Processing data...")
    import time
    time.sleep(2)
    
    # Send COMPLETE event
    print("Sending COMPLETE event...")
    complete_event = RunEvent(
        eventType=RunState.COMPLETE,
        eventTime=datetime.now().isoformat(),
        run=run,
        job=job,
        producer="jupyter-notebook",
        inputs=inputs,
        outputs=outputs
    )
    client.emit(complete_event)
    
    print("\n✓ Custom lineage event created successfully!")
    print(f"\nView in Marquez:")
    print(f"  1. Open http://localhost:3001")
    print(f"  2. Select namespace: datalake")
    print(f"  3. Find job: notebook_custom_job")
    print(f"  4. View your custom lineage!")

# Create sample lineage
create_sample_lineage_event()

## Summary

In this notebook, you learned how to:

1. ✓ Connect to the Marquez API
2. ✓ Query namespaces, jobs, and datasets
3. ✓ Visualize lineage graphs
4. ✓ Track column-level lineage
5. ✓ Perform impact analysis
6. ✓ Create custom lineage events

## Next Steps

- Explore the Marquez Web UI at http://localhost:3001
- Run example Spark jobs and Airflow DAGs
- Create your own lineage-tracked pipelines
- Set up monitoring and alerts
- Integrate lineage into your CI/CD pipeline

## Resources

- [Data Lineage Documentation](../DATA_LINEAGE.md)
- [OpenLineage Documentation](https://openlineage.io/docs/)
- [Marquez API Documentation](https://marquezproject.github.io/marquez/openapi.html)
- [Example Spark Jobs](../spark/apps/lineage_examples/)
- [Example Airflow DAGs](../airflow/dags/lineage_examples/)